[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/04-machine-learning/12_explainability-fairness.ipynb)

*Machine Learning from Scratch · Notebook 12 of 14*

# Explainability, Fairness & Imbalanced Data

Open the black box and check it is fair: linear contributions and Shapley values, permutation importance, partial dependence, counterfactuals, fairness metrics across groups, calibration, thresholds and class weights for imbalanced data, and privacy basics.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Explaining a model is like a teacher showing the marking scheme: every point awarded can be traced, and the same rules apply to every student.

**What interviewers really test here:** Permutation importance vs SHAP, counterfactuals, fairness definitions and their trade-offs, proxies, calibration, and handling class imbalance.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why explain a model at all? *(Basics)*
- [ ] 2. Shapley values: fair credit *(Core)*
- [ ] 3. Global explanations: importance and partial dependence *(Core)*
- [ ] 4. Counterfactuals: what would change the decision? *(Core)*
- [ ] 5. Fairness: measuring bias across groups *(Core)*
- [ ] 6. Calibration and imbalanced data *(Advanced)*
- [ ] 7. Responsible AI in practice *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [ ]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Explanation methods

| Method | Scope | Model | Note |
| --- | --- | --- | --- |
| Coefficients | global + local | linear | scale features first to compare |
| Impurity importance | global | trees | biased to high-cardinality features |
| Permutation importance | global | any | use validation data; correlated features share credit |
| Partial dependence / ICE | global | any | assumes the feature can vary independently |
| SHAP | local → global | any (fast for trees) | additive, consistent |
| LIME | local | any | fits a simple model around one point |
| Counterfactuals | local | any | actionable "what if" |

### Shapley value formula

φᵢ = Σ over subsets S not containing i of |S|! (n − |S| − 1)! / n! × [v(S ∪ {i}) − v(S)]



Properties: **efficiency** (values add up to v(all) − v(none)), **symmetry**, **dummy** (no effect → 0) and **additivity**.

### Fairness metrics

| Metric | Definition |
| --- | --- |
| Demographic parity difference | max − min of P(ŷ=1 \| group) |
| Disparate impact ratio | min ÷ max of P(ŷ=1 \| group); &lt; 0.8 is a red flag |
| Equal opportunity difference | gap in TPR = P(ŷ=1 \| y=1, group) |
| Equalized odds | gaps in both TPR and FPR |
| Predictive parity | equal precision P(y=1 \| ŷ=1, group) |

### Imbalanced classification toolkit

- Metrics: precision, recall, F1, PR-AUC, balanced accuracy — not accuracy.
- Stratified splits so each fold keeps the class ratio.
- Class weights or resampling inside the training fold only.
- Threshold chosen on validation data for the business cost.
- Calibrate if you resampled, because resampling distorts probabilities.

---
# Part 2: Lessons, step by step

## Lesson 1: Why explain a model at all?

*Level: Basics*

A bank model rejects a loan. The customer asks why; the regulator asks whether it discriminates; the data scientist asks whether it learned something real or a quirk of the data. **Explainability** answers all three:




- **Global** explanations: what matters to the model overall (feature importance, partial dependence).
- **Local** explanations: why *this* prediction (contributions, Shapley values, counterfactuals).



Explanations also catch bugs. If "customer ID" is the most important feature, something has leaked.



For a linear model the explanation is exact: each feature contributes coef × (value − average value), and these add up from the average prediction to this prediction.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [ ]:
from sklearn.linear_model import LinearRegression
rng = np.random.default_rng(0)
X = np.round(rng.uniform(0, 10, size=(60, 3)), 1)
y = 3 * X[:, 0] + 1.0 * X[:, 1] + rng.normal(0, 1, 60)


def waterfall():
    m = LinearRegression().fit(X, y)
    row = X[0]
    contrib = m.coef_ * (row - X.mean(axis=0))
    base = m.predict(X).mean()
    fig, ax = plt.subplots(figsize=(5, 2.6))
    labels = ["average"] + [f"x{j}" for j in range(3)] + ["prediction"]
    starts = [0, base, base + contrib[0], base + contrib[:2].sum(), 0]
    heights = [base, *contrib, base + contrib.sum()]
    colors = ["#94a3b8"] + ["#0f766e" if c > 0 else "#b45309" for c in contrib] + ["#334155"]
    ax.bar(labels, heights, bottom=starts, color=colors)
    ax.set_title("From the average prediction to this one")


waterfall()

**✏️ Your turn:** you now have what you need for [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 2: Shapley values: fair credit

*Level: Core*

When features interact, "how much did each one add?" depends on the order you add them. **Shapley values**, from game theory, average a feature's marginal contribution over *every* order. They are the only attribution that is fair in a precise sense: contributions add up to the prediction, identical features get equal credit, and a feature that never matters gets zero.



Exact Shapley values need every subset of features (2ⁿ), so the `shap` library approximates them, or computes them exactly and fast for trees (TreeSHAP). Typical plots: a **waterfall** for one prediction, a **beeswarm** for the whole dataset.



```python
import shap
explainer = shap.Explainer(model, X_background)
sv = explainer(X_test); shap.plots.waterfall(sv[0])
```

**✏️ Your turn:** you now have what you need for [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 3: Global explanations: importance and partial dependence

*Level: Core*

- **Permutation importance**: shuffle one column and see how much the score drops. Works for any model; measure it on validation data. Correlated features share credit, so both can look unimportant.
- **Impurity importance** (trees, forests): total impurity reduction from splits on a feature. Free to compute, but biased towards features with many distinct values.
- **Partial dependence**: set a feature to each value for every row, average the predictions, and plot. It shows the *shape* of the effect — linear, a threshold, a plateau. **ICE** curves show the same per row, revealing interactions.

**✏️ Your turn:** you now have what you need for [Q7](#q7), [Q20](#q20), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 4: Counterfactuals: what would change the decision?

*Level: Core*

"Your loan would be approved if your income were ₹3.13 lakh higher" is the explanation people actually want. A **counterfactual** is the smallest realistic change that flips the outcome. For a linear score w·x + b it has a closed form; for other models it's a search. Good counterfactuals change only things a person can act on — never age or gender.

**✏️ Your turn:** you now have what you need for [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 5: Fairness: measuring bias across groups

*Level: Core*

A model can be accurate overall and still treat groups differently. Compare these across a sensitive attribute (gender, age band, region):



| Criterion | Equal across groups | Check |
| --- | --- | --- |
| Demographic parity | selection rate P(ŷ = 1) | difference, or ratio ≥ 0.8 (the "80% rule") |
| Equal opportunity | true positive rate | qualified people approved equally |
| Equalized odds | TPR and FPR | both kinds of mistakes equal |
| Calibration | P(y = 1 \| score) | a score of 0.7 means 70% in every group |



These can't all hold at once when base rates differ, so choosing one is a policy decision, not a technical one. Dropping the sensitive column doesn't remove bias: **proxies** such as pincode can encode it. Measure, then mitigate: rebalance or reweight training data, add constraints, or set thresholds per group where the law allows.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q10](#q10), [Q11](#q11), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 6: Calibration and imbalanced data

*Level: Advanced*

A model is **calibrated** if, among cases it scores 0.8, about 80% are positive. Check with a reliability curve, the Brier score and ECE; fix with Platt scaling or isotonic regression (`CalibratedClassifierCV`). Calibration matters whenever the probability itself drives a decision or a price.



With **imbalanced classes** (1% fraud), accuracy is useless: predicting "never fraud" scores 99%. Instead:




- Use balanced accuracy, precision/recall, PR-AUC.
- Reweight with `class_weight="balanced"`, or resample (undersample the majority, oversample or SMOTE the minority — on training data only).
- **Move the threshold**: pick it on validation data to maximise F1, or to minimise expected cost when a missed fraud costs far more than a false alarm.

**✏️ Your turn:** you now have what you need for [Q12](#q12), [Q3](#q3), [Q13](#q13), [Q4](#q4), [Q14](#q14), [Q15](#q15), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 7: Responsible AI in practice

*Level: Advanced*

- **Privacy**: remove or mask personal data (names, emails, phone numbers, IDs) before training or sending text to an LLM. Check **k-anonymity**: every combination of quasi-identifiers (pincode, age band, gender) should match at least k people, or individuals can be re-identified.
- **Documentation**: a **model card** records intended use, training data, metrics per group, known limits and ethical considerations; a **datasheet** does the same for a dataset.
- **Human oversight**: high-stakes decisions (credit, hiring, health) need appeal routes and human review; laws such as the EU AI Act and India's DPDP Act set obligations.
- **Monitoring**: fairness and calibration can drift after deployment, so track them like accuracy.

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q17](#q17). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [ ]:
from sklearn.metrics import brier_score_loss, precision_recall_curve, balanced_accuracy_score, r2_score

from sklearn.linear_model import LinearRegression

from sklearn.inspection import permutation_importance, partial_dependence

from sklearn.tree import DecisionTreeRegressor

import math


<a id="q1"></a>
### Q1. Demographic Parity Difference

🟢 Warm-up · *Equal selection rates*

Return `(selection rate per group, largest minus smallest)`, rounded to 4. Selection rate = share predicted positive.

**Examples**

- `group = ['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], pred = [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]` → `({'A': 0.8, 'B': 0.2}, 0.6)`

In [ ]:
def parity(group, pred):
    """
    Return (selection rate per group, largest minus smallest), rounded to 4.
    Selection rate = share predicted positive.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(parity, [
    ((['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]), ({'A': 0.8, 'B': 0.2}, 0.6)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count positives and totals per group.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`groupby(group).mean()` on the predictions.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Count positives and totals per group.

```python
def parity(group, pred):
    pos, tot = {}, {}
    for g, p in zip(group, pred):
        pos[g] = pos.get(g, 0) + p
        tot[g] = tot.get(g, 0) + 1
    rates = {g: round(pos[g] / tot[g], 4) for g in sorted(tot)}
    return (rates, round(max(rates.values()) - min(rates.values()), 4))
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `groupby(group).mean()` on the predictions.

```python
def parity(group, pred):
    r = pd.Series(pred).groupby(pd.Series(group)).mean().round(4)
    return (r.to_dict(), round(r.max() - r.min(), 4))
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count positives and totals per group. That costs O(n) time and O(g) space."

"I can do better: `groupby(group).mean()` on the predictions. That's O(n) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/demographic-parity-difference)

<a id="q2"></a>
### Q2. Disparate Impact and the 80% Rule

🟢 Warm-up · *Ratio of selection rates*

Return `(lowest rate ÷ highest rate, passes the 80% rule)`, ratio rounded to 4.

**Examples**

- `group = ['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], pred = [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]` → `(0.25, False)`
- `group = ['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], pred = [1, 0, 1, 0, 1, 1, 0, 1, 0, 1]` → `(1.0, True)`

In [ ]:
def disparate_impact(group, pred):
    """
    Return (lowest rate ÷ highest rate, passes the 80% rule), ratio rounded to
    4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(disparate_impact, [
    ((['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]), (0.25, False)),
    ((['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], [1, 0, 1, 0, 1, 1, 0, 1, 0, 1]), (1.0, True)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Selection rates in a loop, then divide.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Group means with pandas.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Selection rates in a loop, then divide.

```python
def disparate_impact(group, pred):
    rates = {}
    for g in set(group):
        ps = [p for gg, p in zip(group, pred) if gg == g]
        rates[g] = sum(ps) / len(ps)
    ratio = min(rates.values()) / max(rates.values())
    return (round(ratio, 4), ratio >= 0.8)
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** Group means with pandas.

```python
def disparate_impact(group, pred):
    r = pd.Series(pred).groupby(pd.Series(group)).mean()
    ratio = r.min() / r.max()
    return (round(float(ratio), 4), bool(ratio >= 0.8))
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Selection rates in a loop, then divide. That costs O(n) time and O(g) space."

"I can do better: Group means with pandas. That's O(n) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/disparate-impact-and-the-80-rule)

<a id="q3"></a>
### Q3. Brier Score

🟢 Warm-up · *Probability accuracy*

Return the mean squared difference between predicted probabilities and outcomes, rounded to 4.

**Examples**

- `y = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93]` → `0.1144`

In [ ]:
def brier(y, p):
    """
    Return the mean squared difference between predicted probabilities and
    outcomes, rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(brier, [
    (([0, 0, 0, 1, 0, 1, 1, 1, 1, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93]), 0.1144),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and average (p − y)².

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`brier_score_loss`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Loop and average (p − y)².

```python
def brier(y, p):
    return round(sum((q - t) ** 2 for t, q in zip(y, p)) / len(y), 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `brier_score_loss`.

```python
def brier(y, p):
    return round(float(brier_score_loss(y, p)), 4)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and average (p − y)². That costs O(n) time and O(1) space."

"I can do better: `brier_score_loss`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/brier-score)

<a id="q4"></a>
### Q4. Balanced Class Weights

🟢 Warm-up · *Imbalanced data*

Return the 'balanced' weight for each class (sorted): n\_samples ÷ (n\_classes × class count), rounded to 4.

**Examples**

- `y = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]` → `{0: 0.5556, 1: 5.0}`
- `y = ['a', 'a', 'b', 'c', 'c', 'c']` → `{'a': 1.0, 'b': 2.0, 'c': 0.6667}`

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def class_weights(y):
    """
    Return the 'balanced' weight for each class (sorted): n_samples ÷ (n_classes
    × class count), rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(class_weights, [
    (([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],), {0: 0.5556, 1: 5.0}),
    ((['a', 'a', 'b', 'c', 'c', 'c'],), {'a': 1.0, 'b': 2.0, 'c': 0.6667}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count classes and apply the formula.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`compute_class_weight('balanced', ...)` — the same weights `class_weight='balanced'` uses.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Count classes and apply the formula.

```python
def class_weights(y):
    counts = {}
    for c in y:
        counts[c] = counts.get(c, 0) + 1
    return {c: round(len(y) / (len(counts) * counts[c]), 4) for c in sorted(counts)}
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `compute_class_weight('balanced', ...)` — the same weights `class_weight='balanced'` uses.

```python
def class_weights(y):
    classes = np.unique(y)
    w = compute_class_weight("balanced", classes=classes, y=np.array(y))
    return {c: round(float(v), 4) for c, v in zip(classes.tolist(), w)}
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count classes and apply the formula. That costs O(n) time and O(k) space."

"I can do better: `compute_class_weight('balanced', ...)` — the same weights `class_weight='balanced'` uses. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/balanced-class-weights)

<a id="q5"></a>
### Q5. Accuracy vs Balanced Accuracy

🟢 Warm-up · *The imbalance trap*

Return `(accuracy, balanced accuracy)` rounded to 4. Balanced accuracy averages the recall of each class.

**Examples**

- `y = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], pred = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]` → `(0.95, 0.5)`
- `y = [0, 0, 0, 1, 1], pred = [0, 0, 1, 1, 0]` → `(0.6, 0.5833)`

In [ ]:
def balanced(y, pred):
    """
    Return (accuracy, balanced accuracy) rounded to 4. Balanced accuracy
    averages the recall of each class.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(balanced, [
    (([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), (0.95, 0.5)),
    (([0, 0, 0, 1, 1], [0, 0, 1, 1, 0]), (0.6, 0.5833)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Per-class recall in loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`balanced_accuracy_score`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Per-class recall in loops.

```python
def balanced(y, pred):
    acc = sum(a == b for a, b in zip(y, pred)) / len(y)
    recalls = []
    for c in sorted(set(y)):
        idx = [i for i, a in enumerate(y) if a == c]
        recalls.append(sum(pred[i] == c for i in idx) / len(idx))
    return (round(acc, 4), round(sum(recalls) / len(recalls), 4))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `balanced_accuracy_score`.

```python
def balanced(y, pred):
    return (round(float(np.mean(np.array(y) == np.array(pred))), 4), round(float(balanced_accuracy_score(y, pred)), 4))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Per-class recall in loops. That costs O(n) time and O(k) space."

"I can do better: `balanced_accuracy_score`. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/accuracy-vs-balanced-accuracy)

<a id="q6"></a>
### Q6. Explain a Linear Prediction

🟡 Interview standard · *Contributions (exact SHAP for linear models)*

Fit a linear regression on X, y. For row `i`, return `(base, contributions)` where base = mean prediction and contribution\_j = coef\_j × (x\_ij − mean\_j); they add up to the prediction. Round to 3.

**Examples**

- `X = [[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], y = [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33], i = 0` → `(21.913, [2.867, -2.853, 0.192])`
- `X = [[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], y = [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33], i = 5` → `(21.913, [-10.944, 2.771, -0.012])`

In [ ]:
def linear_contrib(X, y, i):
    """
    Fit a linear regression on X, y. For row i, return (base, contributions)
    where base = mean prediction and contribution_j = coef_j × (x_ij − mean_j);
    they add up to the prediction. Round to 3.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(linear_contrib, [
    (([[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33], 0), (21.913, [2.867, -2.853, 0.192])),
    (([[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33], 5), (21.913, [-10.944, 2.771, -0.012])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Solve the normal equations (XᵀX)β = Xᵀy with an intercept column yourself.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`LinearRegression`, then coef × (row − column means).

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Solve the normal equations (XᵀX)β = Xᵀy with an intercept column yourself.

```python
def linear_contrib(X, y, i):
    A = np.column_stack([np.ones(len(X)), np.array(X)])
    beta = np.linalg.solve(A.T @ A, A.T @ np.array(y))
    coef = beta[1:]
    means = np.array(X).mean(axis=0)
    base = beta[0] + coef @ means
    return (round(float(base), 3), [round(float(c), 3) for c in coef * (np.array(X[i]) - means)])
```

**Cost:** O(nd² + d³) time, O(d²) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `LinearRegression`, then coef × (row − column means).

```python
def linear_contrib(X, y, i):
    X = np.array(X)
    m = LinearRegression().fit(X, y)
    base = m.predict(X).mean()
    return (round(float(base), 3), np.round(m.coef_ * (X[i] - X.mean(axis=0)), 3).tolist())
```

**Cost:** O(nd² + d³) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Solve the normal equations (XᵀX)β = Xᵀy with an intercept column yourself. That costs O(nd² + d³) time and O(d²) space."

"I can do better: `LinearRegression`, then coef × (row − column means). That's O(nd² + d³) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/explain-a-linear-prediction)

<a id="q7"></a>
### Q7. Rank Features by Permutation Importance

🟡 Interview standard · *Model-agnostic importance*

Fit a linear regression, then rank features (most important first) by how much R² drops when that column is shuffled (5 shuffles each, seed 0).

**Examples**

- `X = [[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], y = [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33]` → `[0, 1, 2]`

In [ ]:
def perm_rank(X, y):
    """
    Fit a linear regression, then rank features (most important first) by how
    much R² drops when that column is shuffled (5 shuffles each, seed 0).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(perm_rank, [
    (([[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33]), [0, 1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Shuffle each column yourself with a NumPy generator and re-score.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sklearn.inspection.permutation_importance`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Shuffle each column yourself with a NumPy generator and re-score.

```python
def perm_rank(X, y):
    X, y = np.array(X), np.array(y)
    m = LinearRegression().fit(X, y)
    base = r2_score(y, m.predict(X))
    g = np.random.default_rng(0)
    drops = []
    for j in range(X.shape[1]):
        d = 0
        for _ in range(5):
            Xp = X.copy()
            Xp[:, j] = g.permutation(Xp[:, j])
            d += base - r2_score(y, m.predict(Xp))
        drops.append(d / 5)
    return sorted(range(X.shape[1]), key=lambda j: -drops[j])
```

**Cost:** O(d · r · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `sklearn.inspection.permutation_importance`.

```python
def perm_rank(X, y):
    m = LinearRegression().fit(X, y)
    r = permutation_importance(m, np.array(X), np.array(y), n_repeats=5, random_state=0)
    return [int(j) for j in np.argsort(-r.importances_mean)]
```

**Cost:** O(d · r · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Shuffle each column yourself with a NumPy generator and re-score. That costs O(d · r · n) time and O(n) space."

"I can do better: `sklearn.inspection.permutation_importance`. That's O(d · r · n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/rank-features-by-permutation-importance)

<a id="q8"></a>
### Q8. Partial Dependence

🟡 Interview standard · *Average effect of one feature*

Fit `DecisionTreeRegressor(max_depth=3, random_state=0)`. For each unique value g of feature `j` (sorted), set that column to g for every row and average the predictions. Return the list, rounded to 3.

**Examples**

- `X = [[1, 0], [1, 5], [1, 10], [2, 0], [2, 5], [2, 10], [3, 0], [3, 5], [3, 10], [4, 0], [4, 5], [4, 10]], y = [10, 15, 15, 20, 25, 25, 30, 35, 35, 40, 45, 45], j = 0` → `[13.333, 23.333, 33.333, 43.333]`
- `X = [[1, 0], [1, 5], [1, 10], [2, 0], [2, 5], [2, 10], [3, 0], [3, 5], [3, 10], [4, 0], [4, 5], [4, 10]], y = [10, 15, 15, 20, 25, 25, 30, 35, 35, 40, 45, 45], j = 1` → `[25.0, 30.0, 30.0]`

In [ ]:
def pdp(X, y, j):
    """
    Fit DecisionTreeRegressor(max_depth=3, random_state=0). For each unique
    value g of feature j (sorted), set that column to g for every row and
    average the predictions. Return the list, rounded to 3.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pdp, [
    (([[1, 0], [1, 5], [1, 10], [2, 0], [2, 5], [2, 10], [3, 0], [3, 5], [3, 10], [4, 0], [4, 5], [4, 10]], [10, 15, 15, 20, 25, 25, 30, 35, 35, 40, 45, 45], 0), [13.333, 23.333, 33.333, 43.333]),
    (([[1, 0], [1, 5], [1, 10], [2, 0], [2, 5], [2, 10], [3, 0], [3, 5], [3, 10], [4, 0], [4, 5], [4, 10]], [10, 15, 15, 20, 25, 25, 30, 35, 35, 40, 45, 45], 1), [25.0, 30.0, 30.0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over grid values, overwrite the column, predict, average.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`partial_dependence(model, X, [j], kind='average')`, which uses the unique values as the grid when there are few.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Loop over grid values, overwrite the column, predict, average.

```python
def pdp(X, y, j):
    X = np.array(X, float)
    m = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y)
    out = []
    for g in sorted(set(X[:, j])):
        Xg = X.copy()
        Xg[:, j] = g
        out.append(round(float(m.predict(Xg).mean()), 3))
    return out
```

**Cost:** O(g · n · depth) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `partial_dependence(model, X, [j], kind='average')`, which uses the unique values as the grid when there are few.

```python
def pdp(X, y, j):
    X = np.array(X, float)
    m = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y)
    return np.round(partial_dependence(m, X, [j], kind="average")["average"][0], 3).tolist()
```

**Cost:** O(g · n · depth) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over grid values, overwrite the column, predict, average. That costs O(g · n · depth) time and O(n) space."

"I can do better: `partial_dependence(model, X, [j], kind='average')`, which uses the unique values as the grid when there are few. That's O(g · n · depth) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/partial-dependence)

<a id="q9"></a>
### Q9. Counterfactual: What Would Change the Decision?

🟡 Interview standard · *Actionable explanations*

A logistic model approves when w·x + b ≥ 0. For a rejected applicant, return the smallest increase of feature `j` (rounded up to 0.01) that gets approval.

**Examples**

- `w = [0.8, -1.5, 0.3], b = -4.0, x = [3.0, 1.0, 2.0], j = 0` → `3.13`
- `w = [0.8, -1.5, 0.3], b = -4.0, x = [3.0, 1.0, 2.0], j = 2` → `8.34`

In [ ]:
def counterfactual(w, b, x, j):
    """
    A logistic model approves when w·x + b ≥ 0. For a rejected applicant, return
    the smallest increase of feature j (rounded up to 0.01) that gets approval.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(counterfactual, [
    (([0.8, -1.5, 0.3], -4.0, [3.0, 1.0, 2.0], 0), 3.13),
    (([0.8, -1.5, 0.3], -4.0, [3.0, 1.0, 2.0], 2), 8.34),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Raise the feature in steps of 0.01 until the score reaches 0.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Solve w·x + b + w\_j·δ = 0 for δ, then round up.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Raise the feature in steps of 0.01 until the score reaches 0.

```python
def counterfactual(w, b, x, j):
    steps = 0
    while sum(wi * xi for wi, xi in zip(w, x)) + b + w[j] * steps / 100 < -1e-9:
        steps += 1
    return steps / 100
```

**Cost:** O(Δ / 0.01) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** Solve w·x + b + w\_j·δ = 0 for δ, then round up.

```python
def counterfactual(w, b, x, j):
    gap = -(np.dot(w, x) + b)
    return math.ceil(round(gap / w[j] * 100, 9)) / 100
```

**Cost:** O(d) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Raise the feature in steps of 0.01 until the score reaches 0. That costs O(Δ / 0.01) time and O(1) space."

"I can do better: Solve w·x + b + w\_j·δ = 0 for δ, then round up. That's O(d) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/counterfactual-what-would-change-the-decision)

<a id="q10"></a>
### Q10. Equal Opportunity

🟡 Interview standard · *True positive rate by group*

Among people who truly qualify (y = 1), what share did the model approve in each group? Return `(TPR per group, gap)`, rounded to 4.

**Examples**

- `group = ['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], y = [1, 1, 0, 0, 1, 1, 1, 0, 0, 0], pred = [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]` → `({'A': 1.0, 'B': 0.5}, 0.5)`

In [ ]:
def equal_opportunity(group, y, pred):
    """
    Among people who truly qualify (y = 1), what share did the model approve in
    each group? Return (TPR per group, gap), rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(equal_opportunity, [
    ((['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], [1, 1, 0, 0, 1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]), ({'A': 1.0, 'B': 0.5}, 0.5)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Filter to y = 1 and count per group.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A DataFrame filter and a groupby mean.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Filter to y = 1 and count per group.

```python
def equal_opportunity(group, y, pred):
    hit, tot = {}, {}
    for g, t, p in zip(group, y, pred):
        if t == 1:
            hit[g] = hit.get(g, 0) + p
            tot[g] = tot.get(g, 0) + 1
    tpr = {g: round(hit[g] / tot[g], 4) for g in sorted(tot)}
    return (tpr, round(max(tpr.values()) - min(tpr.values()), 4))
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** A DataFrame filter and a groupby mean.

```python
def equal_opportunity(group, y, pred):
    df = pd.DataFrame({"g": group, "y": y, "p": pred})
    tpr = df[df.y == 1].groupby("g")["p"].mean().round(4)
    return (tpr.to_dict(), round(tpr.max() - tpr.min(), 4))
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Filter to y = 1 and count per group. That costs O(n) time and O(g) space."

"I can do better: A DataFrame filter and a groupby mean. That's O(n) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/equal-opportunity)

<a id="q11"></a>
### Q11. Equalized Odds Gap

🟡 Interview standard · *TPR and FPR together*

Return the larger of the TPR gap and the FPR gap between groups, rounded to 4.

**Examples**

- `group = ['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], y = [1, 1, 0, 0, 1, 1, 1, 0, 0, 0], pred = [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]` → `0.5`

In [ ]:
def equalized_odds(group, y, pred):
    """
    Return the larger of the TPR gap and the FPR gap between groups, rounded to
    4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(equalized_odds, [
    ((['A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B'], [1, 1, 0, 0, 1, 1, 1, 0, 0, 0], [1, 1, 1, 0, 1, 1, 0, 0, 0, 0]), 0.5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Rates for y = 1 and y = 0 per group in loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One groupby over (group, y), then take max − min of each column.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Rates for y = 1 and y = 0 per group in loops.

```python
def equalized_odds(group, y, pred):
    gaps = []
    for label in (1, 0):
        rates = []
        for g in sorted(set(group)):
            ps = [p for gg, t, p in zip(group, y, pred) if gg == g and t == label]
            rates.append(sum(ps) / len(ps))
        gaps.append(max(rates) - min(rates))
    return round(max(gaps), 4)
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** One groupby over (group, y), then take max − min of each column.

```python
def equalized_odds(group, y, pred):
    t = pd.DataFrame({"g": group, "y": y, "p": pred}).groupby(["y", "g"])["p"].mean().unstack()
    return round(float((t.max(axis=1) - t.min(axis=1)).max()), 4)
```

**Cost:** O(n) time, O(g) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Rates for y = 1 and y = 0 per group in loops. That costs O(n) time and O(g) space."

"I can do better: One groupby over (group, y), then take max − min of each column. That's O(n) time and O(g) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/equalized-odds-gap)

<a id="q12"></a>
### Q12. Reliability Curve

🟡 Interview standard · *Calibration*

Split [0, 1] into `n_bins` equal bins. For each non-empty bin return `(mean predicted probability, observed positive rate)`, rounded to 4.

**Examples**

- `y = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], n_bins = 5` → `[(0.1, 0.0), (0.325, 0.5), (0.55, 0.0), (0.7033, 1.0), (0.89, 1.0)]`

In [ ]:
from sklearn.calibration import calibration_curve

def reliability(y, p, n_bins):
    """
    Split [0, 1] into n_bins equal bins. For each non-empty bin return (mean
    predicted probability, observed positive rate), rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reliability, [
    (([0, 0, 0, 1, 0, 1, 1, 1, 1, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], 5), [(0.1, 0.0), (0.325, 0.5), (0.55, 0.0), (0.7033, 1.0), (0.89, 1.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Put each probability in bin ⌊p × n\_bins⌋ and average.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`calibration_curve(y, p, n_bins)` returns both lists.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Put each probability in bin ⌊p × n\_bins⌋ and average.

```python
def reliability(y, p, n_bins):
    sums = {}
    for t, q in zip(y, p):
        b = min(int(q * n_bins), n_bins - 1)
        s = sums.setdefault(b, [0, 0, 0])
        s[0] += q
        s[1] += t
        s[2] += 1
    return [(round(s[0] / s[2], 4), round(s[1] / s[2], 4)) for b, s in sorted(sums.items())]
```

**Cost:** O(n) time, O(bins) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `calibration_curve(y, p, n_bins)` returns both lists.

```python
def reliability(y, p, n_bins):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=n_bins)
    return [(round(float(a), 4), round(float(b), 4)) for a, b in zip(mean_pred, frac_pos)]
```

**Cost:** O(n) time, O(bins) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Put each probability in bin ⌊p × n\_bins⌋ and average. That costs O(n) time and O(bins) space."

"I can do better: `calibration_curve(y, p, n_bins)` returns both lists. That's O(n) time and O(bins) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/reliability-curve)

<a id="q13"></a>
### Q13. Expected Calibration Error

🟡 Interview standard · *One number for calibration*

ECE = Σ over non-empty bins of (bin size ÷ n) × |mean predicted − observed rate|, with `n_bins` equal bins. Round to 4.

**Examples**

- `y = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], n_bins = 5` → `0.221`

In [ ]:
def ece(y, p, n_bins):
    """
    ECE = Σ over non-empty bins of (bin size ÷ n) × |mean predicted − observed
    rate|, with n_bins equal bins. Round to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ece, [
    (([0, 0, 0, 1, 0, 1, 1, 1, 1, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], 5), 0.221),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Accumulate per-bin sums in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`np.digitize` for bins and `np.bincount` for the sums.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Accumulate per-bin sums in a dict.

```python
def ece(y, p, n_bins):
    bins = {}
    for t, q in zip(y, p):
        b = min(int(q * n_bins), n_bins - 1)
        bins.setdefault(b, []).append((q, t))
    total = 0
    for items in bins.values():
        conf = sum(q for q, _ in items) / len(items)
        acc = sum(t for _, t in items) / len(items)
        total += len(items) / len(y) * abs(conf - acc)
    return round(total, 4)
```

**Cost:** O(n) time, O(bins) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `np.digitize` for bins and `np.bincount` for the sums.

```python
def ece(y, p, n_bins):
    y, p = np.array(y), np.array(p)
    b = np.minimum((p * n_bins).astype(int), n_bins - 1)
    cnt = np.bincount(b, minlength=n_bins)
    conf = np.bincount(b, weights=p, minlength=n_bins)
    acc = np.bincount(b, weights=y, minlength=n_bins)
    keep = cnt > 0
    return round(float(np.abs(conf[keep] - acc[keep]).sum() / len(y)), 4)
```

**Cost:** O(n) time, O(bins) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Accumulate per-bin sums in a dict. That costs O(n) time and O(bins) space."

"I can do better: `np.digitize` for bins and `np.bincount` for the sums. That's O(n) time and O(bins) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/expected-calibration-error)

<a id="q14"></a>
### Q14. Threshold That Maximises F1

🟡 Interview standard · *Don't default to 0.5*

Among the predicted probabilities used as thresholds (predict positive if p ≥ t), return `(best threshold, its F1)`, F1 rounded to 4 (lowest threshold on ties).

**Examples**

- `y = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93]` → `(0.35, 0.9231)`
- `y = [0, 0, 0, 0, 0, 0, 0, 1, 0, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93]` → `(0.78, 0.8)`

In [ ]:
def best_f1(y, p):
    """
    Among the predicted probabilities used as thresholds (predict positive if p
    ≥ t), return (best threshold, its F1), F1 rounded to 4 (lowest threshold on
    ties).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(best_f1, [
    (([0, 0, 0, 1, 0, 1, 1, 1, 1, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93]), (0.35, 0.9231)),
    (([0, 0, 0, 0, 0, 0, 0, 1, 0, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93]), (0.78, 0.8)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every threshold and compute precision, recall and F1.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`precision_recall_curve` gives every threshold's precision and recall at once.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Try every threshold and compute precision, recall and F1.

```python
def best_f1(y, p):
    best = None
    for t in sorted(set(p)):
        tp = sum(1 for a, q in zip(y, p) if q >= t and a == 1)
        fp = sum(1 for a, q in zip(y, p) if q >= t and a == 0)
        fn = sum(1 for a, q in zip(y, p) if q < t and a == 1)
        f1 = 2 * tp / (2 * tp + fp + fn) if tp else 0
        if best is None or f1 > best[1] + 1e-12:
            best = (t, f1)
    return (best[0], round(best[1], 4))
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `precision_recall_curve` gives every threshold's precision and recall at once.

```python
def best_f1(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = np.where(prec + rec > 0, 2 * prec * rec / np.maximum(prec + rec, 1e-12), 0)[:-1]
    i = max(range(len(thr)), key=lambda k: (round(f1[k], 12), -thr[k]))
    return (float(thr[i]), round(float(f1[i]), 4))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every threshold and compute precision, recall and F1. That costs O(n²) time and O(1) space."

"I can do better: `precision_recall_curve` gives every threshold's precision and recall at once. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/threshold-that-maximises-f1)

<a id="q15"></a>
### Q15. Cost-Sensitive Threshold

🟡 Interview standard · *Errors have different prices*

A missed fraud costs `c_fn`, a false alarm costs `c_fp`. Among thresholds 0.05, 0.10, …, 0.95 return `(best threshold, total cost)` (lowest threshold on ties).

**Examples**

- `y = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], c_fp = 1, c_fn = 10` → `(0.35, 1)`
- `y = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1], p = [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], c_fp = 5, c_fn = 1` → `(0.6, 1)`

In [ ]:
def cost_threshold(y, p, c_fp, c_fn):
    """
    A missed fraud costs c_fn, a false alarm costs c_fp. Among thresholds 0.05,
    0.10, …, 0.95 return (best threshold, total cost) (lowest threshold on
    ties).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(cost_threshold, [
    (([0, 0, 0, 1, 0, 1, 1, 1, 1, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], 1, 10), (0.35, 1)),
    (([0, 0, 0, 1, 0, 1, 1, 1, 1, 1], [0.05, 0.15, 0.3, 0.35, 0.55, 0.62, 0.71, 0.78, 0.85, 0.93], 5, 1), (0.6, 1)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over thresholds and rows counting the two error types.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Broadcast thresholds against probabilities to get every cost at once.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Loop over thresholds and rows counting the two error types.

```python
def cost_threshold(y, p, c_fp, c_fn):
    best = None
    for k in range(1, 20):
        t = round(k * 0.05, 2)
        cost = 0
        for a, q in zip(y, p):
            if q >= t and a == 0:
                cost += c_fp
            elif q < t and a == 1:
                cost += c_fn
        if best is None or cost < best[1]:
            best = (t, cost)
    return best
```

**Cost:** O(tn) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** Broadcast thresholds against probabilities to get every cost at once.

```python
def cost_threshold(y, p, c_fp, c_fn):
    t = np.round(np.arange(1, 20) * 0.05, 2)
    y, p = np.array(y), np.array(p)
    pred = p[None, :] >= t[:, None]
    cost = c_fp * (pred & (y == 0)).sum(axis=1) + c_fn * (~pred & (y == 1)).sum(axis=1)
    i = int(cost.argmin())
    return (float(t[i]), int(cost[i]))
```

**Cost:** O(tn) time, O(tn) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over thresholds and rows counting the two error types. That costs O(tn) time and O(1) space."

"I can do better: Broadcast thresholds against probabilities to get every cost at once. That's O(tn) time and O(tn) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/cost-sensitive-threshold)

<a id="q16"></a>
### Q16. Mask Personal Data

🟡 Interview standard · *Privacy before training*

Replace email addresses with `[EMAIL]` and 10-digit phone numbers (optionally starting +91) with `[PHONE]`.

**Examples**

- `text = 'Call 9876543210 or mail asha.k@mail.com'` → `'Call [PHONE] or mail [EMAIL]'`
- `text = '+91 9123456789, ravi@x.in, order 12345'` → `'[PHONE], [EMAIL], order 12345'`

In [ ]:
import re

def mask_pii(text):
    """
    Replace email addresses with [EMAIL] and 10-digit phone numbers (optionally
    starting +91) with [PHONE].
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mask_pii, [
    (('Call 9876543210 or mail asha.k@mail.com',), 'Call [PHONE] or mail [EMAIL]'),
    (('+91 9123456789, ravi@x.in, order 12345',), '[PHONE], [EMAIL], order 12345'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan word by word: '@' with a '.' after it is an email; strip punctuation and check for 10 digits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two regular expressions with `re.sub`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Scan word by word: '@' with a '.' after it is an email; strip punctuation and check for 10 digits.

```python
def mask_pii(text):
    words = text.split(" ")
    out = []
    skip = False
    for i, w in enumerate(words):
        if skip:
            skip = False
            continue
        core = w.rstrip(",.")
        tail = w[len(core):]
        if "@" in core and "." in core.split("@")[-1]:
            out.append("[EMAIL]" + tail)
        elif core == "+91" and i + 1 < len(words) and words[i + 1].rstrip(",.").isdigit() and len(words[i + 1].rstrip(",.")) == 10:
            nxt = words[i + 1]
            out.append("[PHONE]" + nxt[len(nxt.rstrip(",.")):])
            skip = True
        elif core.isdigit() and len(core) == 10:
            out.append("[PHONE]" + tail)
        else:
            out.append(w)
    return " ".join(out)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** Two regular expressions with `re.sub`.

```python
def mask_pii(text):
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "[EMAIL]", text)
    return re.sub(r"(\+91\s?)?\b\d{10}\b", "[PHONE]", text)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan word by word: '@' with a '.' after it is an email; strip punctuation and check for 10 digits. That costs O(n) time and O(n) space."

"I can do better: Two regular expressions with `re.sub`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/mask-personal-data)

<a id="q17"></a>
### Q17. k-Anonymity

🟡 Interview standard · *Re-identification risk*

Generalise age to its decade (34 → '30s') and return the size of the smallest group of identical (pincode, decade, gender) rows — the dataset's k.

**Examples**

- `rows = [('411001', 34, 'F'), ('411001', 35, 'F'), ('411002', 34, 'M'), ('411001', 36, 'F'), ('411002', 31, 'M')]` → `2`
- `rows = [('411001', 34, 'F'), ('411001', 35, 'F'), ('411002', 34, 'M'), ('411001', 36, 'F'), ('411002', 31, 'M'), ('411003', 52, 'F')]` → `1`

In [ ]:
def k_anonymity(rows):
    """
    Generalise age to its decade (34 → '30s') and return the size of the
    smallest group of identical (pincode, decade, gender) rows — the dataset's
    k.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(k_anonymity, [
    (([('411001', 34, 'F'), ('411001', 35, 'F'), ('411002', 34, 'M'), ('411001', 36, 'F'), ('411002', 31, 'M')],), 2),
    (([('411001', 34, 'F'), ('411001', 35, 'F'), ('411002', 34, 'M'), ('411001', 36, 'F'), ('411002', 31, 'M'), ('411003', 52, 'F')],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build keys and count in a dict.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`groupby([...]).size().min()`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Build keys and count in a dict.

```python
def k_anonymity(rows):
    counts = {}
    for pin, age, sex in rows:
        key = (pin, str(age // 10 * 10) + "s", sex)
        counts[key] = counts.get(key, 0) + 1
    return min(counts.values())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `groupby([...]).size().min()`.

```python
def k_anonymity(rows):
    df = pd.DataFrame(rows, columns=["pin", "age", "sex"])
    df["decade"] = (df["age"] // 10 * 10).astype(str) + "s"
    return int(df.groupby(["pin", "decade", "sex"]).size().min())
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build keys and count in a dict. That costs O(n) time and O(n) space."

"I can do better: `groupby([...]).size().min()`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/k-anonymity)

<a id="q18"></a>
### Q18. Find Proxy Features

🟡 Interview standard · *Bias sneaks back in*

Dropping a sensitive column isn't enough if other columns predict it. Return features whose |correlation| with the sensitive column is above `limit`, sorted.

**Examples**

- `features = {'pincode_income': [9, 8, 9, 3, 2, 3], 'years_exp': [5, 2, 7, 4, 6, 3], 'gym_member': [1, 1, 1, 0, 0, 1]}, sensitive = [1, 1, 1, 0, 0, 0], limit = 0.7` → `['gym_member', 'pincode_income']`

In [ ]:
def proxies(features, sensitive, limit):
    """
    Dropping a sensitive column isn't enough if other columns predict it. Return
    features whose |correlation| with the sensitive column is above limit,
    sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(proxies, [
    (({'pincode_income': [9, 8, 9, 3, 2, 3], 'years_exp': [5, 2, 7, 4, 6, 3], 'gym_member': [1, 1, 1, 0, 0, 1]}, [1, 1, 1, 0, 0, 0], 0.7), ['gym_member', 'pincode_income']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Pearson r for each column by formula.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`DataFrame.corrwith`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Pearson r for each column by formula.

```python
def proxies(features, sensitive, limit):
    out = []
    s = sensitive
    ms = sum(s) / len(s)
    for name, x in features.items():
        mx = sum(x) / len(x)
        sxy = sum((a - mx) * (b - ms) for a, b in zip(x, s))
        r = sxy / math.sqrt(sum((a - mx) ** 2 for a in x) * sum((b - ms) ** 2 for b in s))
        if abs(r) > limit:
            out.append(name)
    return sorted(out)
```

**Cost:** O(dn) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `DataFrame.corrwith`.

```python
def proxies(features, sensitive, limit):
    r = pd.DataFrame(features).corrwith(pd.Series(sensitive))
    return sorted(r[r.abs() > limit].index)
```

**Cost:** O(dn) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Pearson r for each column by formula. That costs O(dn) time and O(1) space."

"I can do better: `DataFrame.corrwith`. That's O(dn) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/find-proxy-features)

<a id="q19"></a>
### Q19. Shapley Values by Hand

🔴 Stretch · *Fair credit for each feature*

`v` maps every sorted tuple of players to the value that team earns. Return each player's Shapley value (sorted by player), rounded to 4.

**Examples**

- `v = {(): 0, ('a',): 10, ('b',): 20, ('c',): 5, ('a', 'b'): 40, ('a', 'c'): 15, ('b', 'c'): 25, ('a', 'b', 'c'): 50}` → `{'a': 16.6667, 'b': 26.6667, 'c': 6.6667}`

In [ ]:
from itertools import permutations, combinations

def shapley(v):
    """
    v maps every sorted tuple of players to the value that team earns. Return
    each player's Shapley value (sorted by player), rounded to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shapley, [
    (({(): 0, ('a',): 10, ('b',): 20, ('c',): 5, ('a', 'b'): 40, ('a', 'c'): 15, ('b', 'c'): 25, ('a', 'b', 'c'): 50},), {'a': 16.6667, 'b': 26.6667, 'c': 6.6667}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Average each player's marginal contribution over every ordering of the players.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The subset formula: Σ |S|!(n−|S|−1)!/n! × (v(S ∪ {i}) − v(S)).

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Average each player's marginal contribution over every ordering of the players.

```python
def shapley(v):
    players = sorted(max(v, key=len))
    total = {p: 0.0 for p in players}
    orders = list(permutations(players))
    for order in orders:
        team = []
        for p in order:
            before = v[tuple(sorted(team))]
            team.append(p)
            total[p] += v[tuple(sorted(team))] - before
    return {p: round(total[p] / len(orders), 4) for p in players}
```

**Cost:** O(n! · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** The subset formula: Σ |S|!(n−|S|−1)!/n! × (v(S ∪ {i}) − v(S)).

```python
def shapley(v):
    players = sorted(max(v, key=len))
    n = len(players)
    out = {}
    for p in players:
        others = [q for q in players if q != p]
        phi = 0.0
        for size in range(n):
            w = math.factorial(size) * math.factorial(n - size - 1) / math.factorial(n)
            for S in combinations(others, size):
                phi += w * (v[tuple(sorted(S + (p,)))] - v[S])
        out[p] = round(phi, 4)
    return out
```

**Cost:** O(2ⁿ · n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Average each player's marginal contribution over every ordering of the players. That costs O(n! · n) time and O(n) space."

"I can do better: The subset formula: Σ |S|!(n−|S|−1)!/n! × (v(S ∪ {i}) − v(S)). That's O(2ⁿ · n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/shapley-values-by-hand)

<a id="q20"></a>
### Q20. Where Tree Importances Come From

🔴 Stretch · *Impurity decrease*

Fit `DecisionTreeRegressor(max_depth=3, random_state=0)` and return each feature's importance, computed from the tree: total (weighted) impurity decrease from its splits, normalised to sum to 1. Round to 4.

**Examples**

- `X = [[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], y = [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33]` → `[0.9534, 0.0466, 0.0]`
- `X = [[1, 0], [1, 5], [1, 10], [2, 0], [2, 5], [2, 10], [3, 0], [3, 5], [3, 10], [4, 0], [4, 5], [4, 10]], y = [10, 15, 15, 20, 25, 25, 30, 35, 35, 40, 45, 45]` → `[0.9574, 0.0426]`

In [ ]:
def tree_importance(X, y):
    """
    Fit DecisionTreeRegressor(max_depth=3, random_state=0) and return each
    feature's importance, computed from the tree: total (weighted) impurity
    decrease from its splits, normalised to sum to 1. Round to 4.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tree_importance, [
    (([[6.4, 2.7, 0.4], [0.2, 8.1, 9.1], [6.1, 7.3, 5.4], [9.4, 8.2, 0.0], [8.6, 0.3, 7.3], [1.8, 8.6, 5.4], [3.0, 4.2, 0.3], [1.2, 6.7, 6.5], [6.2, 3.8, 10.0], [9.8, 6.9, 6.5], [6.9, 3.9, 1.4], [7.2, 5.3, 3.1], [4.9, 8.9, 9.3], [3.6, 5.7, 3.2], [5.9, 3.4, 3.9], [8.9, 2.3, 6.2], [0.8, 8.3, 7.9], [2.4, 8.8, 0.6], [3.4, 1.5, 4.5], [8.0, 2.3, 0.5], [4.0, 2.0, 0.9], [5.8, 3.0, 6.7], [2.0, 9.4, 3.7], [1.1, 6.3, 9.3], [4.4, 9.5, 5.0], [4.3, 6.2, 10.0], [9.5, 4.6, 7.6], [5.0, 5.3, 7.9], [4.1, 7.3, 7.1], [9.3, 1.1, 7.3], [9.3, 9.7, 0.1], [8.6, 9.8, 9.6], [1.5, 9.7, 8.9], [8.2, 4.8, 2.3], [8.0, 9.2, 2.7], [5.4, 4.4, 9.3], [0.4, 7.3, 6.1], [0.3, 7.2, 0.2], [7.6, 5.1, 9.3], [0.7, 8.4, 0.7], [3.4, 4.3, 9.7], [5.6, 2.6, 2.4], [8.9, 2.3, 1.2], [2.9, 5.9, 5.5], [8.1, 5.6, 2.9], [4.1, 8.2, 6.3], [9.6, 3.7, 5.5], [5.9, 8.5, 1.5], [4.1, 9.1, 0.4], [8.2, 4.2, 8.3], [0.1, 3.7, 0.8], [6.5, 2.7, 7.0], [9.4, 1.3, 8.6], [0.6, 3.8, 4.3], [4.9, 9.8, 7.8], [3.1, 2.7, 8.6], [8.8, 5.1, 3.4], [9.9, 3.2, 1.8], [8.8, 8.1, 6.7], [9.6, 9.3, 7.5]], [22.93, 6.77, 25.36, 36.2, 25.06, 14.61, 13.0, 9.86, 22.92, 35.82, 25.99, 27.25, 23.13, 14.56, 19.79, 30.09, 10.65, 15.72, 13.34, 25.02, 13.41, 19.93, 15.99, 8.94, 22.09, 17.49, 33.83, 21.11, 19.12, 29.16, 36.31, 35.13, 15.58, 29.54, 35.51, 19.81, 9.08, 7.9, 28.47, 10.49, 13.94, 18.53, 32.07, 14.52, 27.88, 19.85, 33.18, 25.7, 22.76, 29.8, 3.85, 21.73, 28.5, 4.9, 23.03, 13.2, 33.09, 31.64, 33.32, 36.33]), [0.9534, 0.0466, 0.0]),
    (([[1, 0], [1, 5], [1, 10], [2, 0], [2, 5], [2, 10], [3, 0], [3, 5], [3, 10], [4, 0], [4, 5], [4, 10]], [10, 15, 15, 20, 25, 25, 30, 35, 35, 40, 45, 45]), [0.9574, 0.0426]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the tree arrays: for each split add w·imp − w\_left·imp\_left − w\_right·imp\_right to its feature.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`model.feature_importances_`.

</details>

<details>
<summary><b>✅ Answer: From scratch — plain Python & NumPy</b></summary>

**Idea:** Walk the tree arrays: for each split add w·imp − w\_left·imp\_left − w\_right·imp\_right to its feature.

```python
def tree_importance(X, y):
    m = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y)
    t = m.tree_
    imp = [0.0] * len(X[0])
    for node in range(t.node_count):
        left, right = t.children_left[node], t.children_right[node]
        if left == -1:
            continue
        w, wl, wr = t.weighted_n_node_samples[node], t.weighted_n_node_samples[left], t.weighted_n_node_samples[right]
        imp[t.feature[node]] += w * t.impurity[node] - wl * t.impurity[left] - wr * t.impurity[right]
    total = sum(imp)
    return [round(v / total, 4) for v in imp]
```

**Cost:** O(nodes) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: With scikit-learn</b></summary>

**Idea:** `model.feature_importances_`.

```python
def tree_importance(X, y):
    m = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X, y)
    return np.round(m.feature_importances_, 4).tolist()
```

**Cost:** O(1) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the tree arrays: for each split add w·imp − w\_left·imp\_left − w\_right·imp\_right to its feature. That costs O(nodes) time and O(d) space."

"I can do better: `model.feature_importances_`. That's O(1) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ml/#/q/explainability-fairness/where-tree-importances-come-from)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Permutation importance vs impurity importance?</b></summary>

Permutation importance measures the drop in a chosen score when a feature is shuffled, works for any model and on held-out data. Impurity importance is computed from training splits of trees and favours features with many distinct values.

</details>

<details>
<summary><b>2. What are SHAP values?</b></summary>

Shapley values from game theory applied to features: each feature's average marginal contribution over all orders of adding features. They add up exactly from the average prediction to this prediction.

</details>

<details>
<summary><b>3. Can a model be fair if it doesn't use gender?</b></summary>

Not necessarily. Other features (pincode, name, shopping habits) can act as proxies. Fairness must be measured on outcomes per group.

</details>

<details>
<summary><b>4. Why can't all fairness criteria be satisfied together?</b></summary>

When groups have different base rates, a calibrated model can't also have equal false-positive and false-negative rates across groups (except for a perfect model). You have to choose which notion matters for the decision.

</details>

<details>
<summary><b>5. How do you handle a 1:99 class imbalance?</b></summary>

Use precision/recall or PR-AUC, stratified splits, class weights or resampling on training data only, and pick the decision threshold from validation data using the real costs of each error.

</details>

<details>
<summary><b>6. What does it mean for a model to be calibrated?</b></summary>

Its predicted probabilities match observed frequencies: among cases scored 0.3, about 30% are positive. Check with reliability curves, Brier score or ECE.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why explain a model at all?
- [ ] I can explain: Shapley values: fair credit
- [ ] I can explain: Global explanations: importance and partial dependence
- [ ] I can explain: Counterfactuals: what would change the decision?
- [ ] I can explain: Fairness: measuring bias across groups
- [ ] I can explain: Calibration and imbalanced data
- [ ] I can explain: Responsible AI in practice

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Mask Personal Data
- [ ] [Q17](#q17) k-Anonymity
- [ ] [Q18](#q18) Find Proxy Features
- [ ] [Q19](#q19) Shapley Values by Hand
- [ ] [Q20](#q20) Where Tree Importances Come From

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.